In [3]:
# 기본세팅 : 키 확인, 클라이언트 생성 및 모델 설정
import os
from openai import OpenAI
os.getenv("OPENAI_API_KEY")[:8] # 키 확인

client = OpenAI()
API_MODEL = "gpt-5.6-luna"

In [ ]:
r = client.responses.create(
    model=API_MODEL,
    input="안녕, 자기소개 해봐. 한 문장으로."  # messages -> input
)

r.output_text   # <- choices[0].massage.content 대신에 쉽게 접근

'안녕하세요, 궁금한 것을 함께 풀어가는 AI 어시스턴트입니다.'

In [7]:
print(r.output_text, r.id) # 다른 점
print(r.usage.input_tokens, r.usage.output_tokens)  # usage는 token 사용은 같다.

안녕하세요, 궁금한 것을 함께 풀어가는 AI 어시스턴트입니다. resp_0d6478c27ef2b927006a84f7461788819a8c5fd553524bf107
19 23


# 기억

In [ ]:
r = client.responses.create(
    model=API_MODEL,
    input="안녕, 내 이름은 길동이야."
)
print(r.output_text)
r = client.responses.create(
    model=API_MODEL,
    input="내 이름이 뭐라고 했지?"
)
print(r.output_text)
# 각 API 간은 무상태성이라, 상태가 이어지지 않고, 기억 X

안녕하세요, 길동님! 만나서 반가워요. 무엇을 도와드릴까요?
아직 이름을 말씀해 주시지 않았어요.


In [ ]:
r = client.responses.create(
    model=API_MODEL,
    input="안녕, 내 이름은 길동이야."
)
print(r.output_text)
r = client.responses.create(
    model=API_MODEL,
    input="내 이름이 뭐라고 했지?",
    previous_response_id=r.id    # 이전 응답 ID를 매개변수로 주면 => 기억
)
print(r.output_text)

안녕하세요, 길동님! 만나서 반가워요. 무엇을 도와드릴까요?
길동이라고 하셨어요.


In [ ]:
prev = None
for i, q in enumerate(["내 이름은 금이야. 취미는 등산이고 서울에 살아.",
                       "내 취미가 뭐랬지?", "내가 어디 산다고 했지?", "내 이름은?"], 1):
    rr = client.responses.create(model=API_MODEL, input=q, previous_response_id=prev)
    prev = rr.id
    print(f"{i}번째턴, 입력토큰 : {rr.usage.input_tokens}, 출력토큰 : {rr.usage.output_tokens}, 내용 : {rr.output_text} ")
# API 메시지 배열이 안보여도 멀티턴 입력 토큰 누적량은 같다.

1번째턴, 입력토큰 : 23, 출력토큰 : 105, 내용 : 반가워요, 금님! 서울에 살면서 등산을 즐기시는군요. 북한산, 관악산, 청계산처럼 좋은 산이 많아 취미와 잘 어울리네요. 앞으로 금님이라고 부를게요. 
2번째턴, 입력토큰 : 143, 출력토큰 : 14, 내용 : 금님의 취미는 등산이에요. 
3번째턴, 입력토큰 : 171, 출력토큰 : 14, 내용 : 금님은 서울에 산다고 했어요. 
4번째턴, 입력토큰 : 195, 출력토큰 : 11, 내용 : 이름은 금이에요. 


- 차이점 :
  - chat : 배열을 내가 들고 있고 -> (리스트를 내가 조작 가능하다)
    - 맥락을 다른 LLM 제공 업체로 옮길 수 있음
  - responses : 배열을 서버가 들고 있다. -> (리스트 조작이 불가능)
    - 맥락을 내가 소유하고 있지 않다.

In [ ]:
# 맥락을 서버에 저장하지 않고 싶을 경우
r = client.responses.create(
    model=API_MODEL,
    input="안녕, hello",
    store=False, # 서버에 안 남는다.
)
print(r.id)
print(r.output_text)

resp_05db7ede3e8131f5016a84fb734de8819aa7d166a1d27f8383
안녕! Hello! 😊 무엇을 도와드릴까요?


In [ ]:
try:
    r = client.responses.create(
        model=API_MODEL,
        input="내가 아까 뭐라고 했어?",
        previous_response_id=r.id  #    store=False 했던 id는 아예 사용 안됨
    )
except Exception as e:
    print(str(e))
# Error code: 400 - {'error': {'message': "Previous response with id .. not found

Error code: 400 - {'error': {'message': "Previous response with id 'resp_05db7ede3e8131f5016a84fb734de8819aa7d166a1d27f8383' not found.", 'type': 'invalid_request_error', 'param': 'previous_response_id', 'code': 'previous_response_not_found'}}


## 시스템 프롬프트는?

In [15]:
r = client.responses.create(
    model=API_MODEL,
    instructions="나는 초등학교 선생님이야. 쉽게 설명해줘",
    input="토큰이 뭐니?",
)
print(r.id)
print(r.output_text)

resp_077742461ed09668006a84fca71578819a86ff486594281917
**토큰(token)**은 컴퓨터가 글을 읽고 처리할 때 사용하는 **작은 글자 조각**이에요.

예를 들어,

> “나는 학교에 가요.”

라는 문장을 컴퓨터는 단어 또는 글자 일부로 나누어 여러 개의 토큰으로 볼 수 있어요.

쉽게 말하면:

- 사람에게 글 = 문장과 단어
- 컴퓨터에게 글 = 토큰들의 묶음

ChatGPT에서는 질문과 답변이 토큰으로 계산돼요. 그래서 **글이 길수록 토큰도 많아지고**, 사용할 수 있는 토큰 수에는 한도가 있을 수 있어요.

참고로 토큰은 상황에 따라 **게임용 표**, **교환권**, **인증용 번호**라는 뜻으로도 사용돼요.


## 두 API 응답 객체 비교

In [16]:
rc = client.chat.completions.create(
    model=API_MODEL,
    messages=[{"role":"user","content":"한 단어로 인사해"}]
)

rr = client.responses.create(
    model=API_MODEL,
    input="한 단어로 인사해",
)

In [ ]:
# API 응답 객체 구조와 키 수도 많이 다름.
print(rr)
print(rc)
print(len(rr.model_dump().keys()), rr.model_dump().keys()) # 39
print(len(rc.model_dump().keys()), rc.model_dump().keys()) # 9

Response(id='resp_0d8b0824b631147c006a8501163e54819ba9e1c8140922d554', created_at=1787101462.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.6-luna', object='response', output=[ResponseOutputMessage(id='msg_0d8b0824b631147c006a85011763ac819b847b2f4c0c856028', content=[ResponseOutputText(annotations=[], text='안녕하세요', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=0.98, background=False, completed_at=1787101463.0, conversation=None, max_output_tokens=None, max_tool_calls=None, moderation=None, previous_response_id=None, prompt=None, prompt_cache_key=None, prompt_cache_options=None, prompt_cache_retention='24h', reasoning=Reasoning(context='all_turns', effort='medium', generate_summary=None, mode='standard', summary=None), safety_identifier=None, service_tier='default', status='completed', text=ResponseTextCon

In [ ]:
def keys_of(obj):
    """응답 객체의 최상위 키를 줄 맞춰 찍는다 (39개라 한 줄로 보면 안 읽힌다)"""
    ks = sorted(obj.model_dump().keys())
    for i in range(0, len(ks), 6):
        print("   ", "  ".join(f"{k:22s}" for k in ks[i:i + 6]).rstrip())
    print(f"    → 모두 {len(ks)}개")

print("[chat] 최상위 키")
keys_of(rc)
print()
print("[resp] 최상위 키")
keys_of(rr)
# responses api 응답 객체는 서버에 저장된 객체

[chat] 최상위 키
    choices                 created                 id                      model                   moderation              object
    service_tier            system_fingerprint      usage
    → 모두 9개

[resp] 최상위 키
    background              billing                 completed_at            conversation            created_at              error
    frequency_penalty       id                      incomplete_details      instructions            max_output_tokens       max_tool_calls
    metadata                model                   moderation              object                  output                  parallel_tool_calls
    presence_penalty        previous_response_id    prompt                  prompt_cache_key        prompt_cache_options    prompt_cache_retention
    reasoning               safety_identifier       service_tier            status                  store                   temperature
    text                    tool_choice             tool_usage              

In [ ]:
# 1. C
rr = client.responses.create(
    model=API_MODEL,
    input="안녕 내 이름은 장원이야.",
)
# 2. R
obj = client.responses.retrieve(rr.id)  # 이전 요청-응답 객체를 검색

In [28]:
obj.output_text, obj.usage.input_tokens

('안녕하세요, 장원님! 만나서 반가워요. 무엇을 도와드릴까요?', 15)

In [ ]:
# 3. DELETE
client.responses.delete(rr.id)   # 서버에서 응답 객체를 지우기.

In [31]:
try:
    obj = client.responses.retrieve(rr.id) # 삭제된 객체는 서버에서 조회 불가
except Exception as e:
    print(e)

Error code: 404 - {'error': {'message': "Response with id 'resp_0da230298efbba70006a8502aca66087d0a0750e36bf920937' not found.", 'type': 'invalid_request_error', 'param': None, 'code': None}}


In [36]:
rc.choices[0].model_dump()  # chat
rr.output # responses # 답이 담기는 곳의 그릇도 다 다르다.

[ResponseOutputMessage(id='msg_0da230298efbba70006a8502ad2c4487d08105745c1c73afcd', content=[ResponseOutputText(annotations=[], text='안녕하세요, 장원님! 만나서 반가워요. 무엇을 도와드릴까요?', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')]

In [39]:
rr = client.responses.create(
    model=API_MODEL,
    input="국제 관계에 대해서 설명해줘",
    max_output_tokens=20  # 최대토큰, 파라미터 명이 다름.
)
print(rr.output_text)  # 출력이 안되고
print(rr.status)       # 안된 상태
print(rr.incomplete_details)  # 안된 이유에 대해 자세히


incomplete
IncompleteDetails(reason='max_output_tokens')


In [ ]:
# messages list를 입력하듯이 입력하는 방법 => 가능
rr = client.responses.create(
    model=API_MODEL,
    input=[{"role":"developer", "content":"너는 한 단어로만 대답한다."},
           {"role":"user", "content":"브라질의 수도는?"}],
)
print(rr.output_text)  # 출력이 안되고

브라질리아


## 구조화된 출력

In [ ]:
# 구조화된 출력 등 그대로 사용되지만, key, 매개변수 명 주의
from pydantic import BaseModel

class ReviewAnalysis(BaseModel):
    sentiment: str
    rating: int
    summary: str

rp = client.responses.parse(
    model=API_MODEL,
    input="이 리뷰를 분석해 줘: 배송은 느렸지만 물건은 기대 이상. 또 살 듯.",
    text_format=ReviewAnalysis,          # 어제는 response_format= 이었다. 이름만 다르다
)
print(rp.output_parsed)

sentiment='긍정적' rating=4 summary='배송은 느렸지만 상품 품질이 기대 이상이어서 전반적으로 만족했으며, 재구매 의향이 있습니다.'


## 툴 콜링

In [42]:
import json
import urllib


def get_weather(city: str, latitude: float, longitude: float) -> str:
    """그 좌표의 지금 날씨를 open-meteo 에서 가져온다 (키 불필요)"""
    url = (f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}"
           f"&current=temperature_2m,relative_humidity_2m,wind_speed_10m&timezone=auto")
    with urllib.request.urlopen(url, timeout=10) as resp:
        c = json.load(resp)["current"]
    return (f"{city} 기온 {c['temperature_2m']}도, 습도 {c['relative_humidity_2m']}%, "
            f"바람 {c['wind_speed_10m']}m/s")

print(get_weather("서울", 37.5665, 126.9780))

서울 기온 26.6도, 습도 84%, 바람 3.1m/s


In [43]:
tools = [{
    "type": "function",
    "name": "get_weather",                      # 어제는 function 안에 있었다
    "description": "특정 도시의 현재 날씨(기온·습도·바람)를 가져온다. 실시간 정보가 필요할 때 쓴다.",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {"type": "string", "description": "도시 이름"},
            "latitude": {"type": "number", "description": "위도"},
            "longitude": {"type": "number", "description": "경도"},
        },
        "required": ["city", "latitude", "longitude"],
        "additionalProperties": False,          # 어제 함정 B 에서 배운 그 줄
    },
}]

In [45]:
r = client.responses.create(
    model=API_MODEL,
    input="지금 부산 습도 얼마니?",
    tools=tools # <-
)

In [ ]:
r.output_text
r.output[0] # 추론 아이템
r.output[1] # 함수 호출

ResponseFunctionToolCall(arguments='{"city":"부산","latitude":35.1796,"longitude":129.0756}', call_id='call_hBULcZuwmzlvT7wzasrtHVA8', name='get_weather', type='function_call', id='fc_0dc3aa65eb49a0c3006a85073e535887d0bce72d3ec503e1cc', caller=None, namespace=None, status='completed')

In [54]:
r.output[1].arguments, r.output[1].name
args = json.loads(r.output[1].arguments)
result = get_weather(**args)
print(result)

부산 기온 28.6도, 습도 81%, 바람 7.9m/s


In [ ]:
fc = r.output[1]   # 이전 출력에서 펑션 콜 객체 꺼내기
args = json.loads(fc.arguments)          # 모델이 준 인자는 '문자열'이다. 파이썬 값으로 되돌린다
result = get_weather(**args)             # ← 여기서 실제로 실행된다. 실시간 날씨는 이 줄에서 나온다
print("내 코드의 실행 결과 :", result)

r2 = client.responses.create(
    model=API_MODEL,
    previous_response_id=r.id,           # 앞 대화는 서버가 들고 있다
    tools=tools,
    input=[{                             # 새로 보내는 것은 '결과 조각' 하나뿐
        "type": "function_call_output",  # 어제의 {"role": "tool", ...} 자리
        "call_id": fc.call_id,           # 어느 요청에 대한 답인지 짝을 짓는다
        "output": result,
    }],
)
print("최종 답 :", r2.output_text)

내 코드의 실행 결과 : 부산 기온 28.6도, 습도 81%, 바람 7.9m/s
최종 답 : 현재 부산의 습도는 **81%**입니다.


In [ ]:
r = client.responses.create(
    model=API_MODEL,
    input="지금 부산 습도 얼마니?",
)
print(r.output_text)

r = client.responses.create(
    model=API_MODEL,
    input="지금 부산 습도 얼마니?",
    tools=[{"type": "web_search"}]  # OpenAI 내장 웹 검색 도구
)
print(r.output_text)

실시간 날씨 정보에 접속할 수 없어 현재 부산 습도를 확인해 드리기 어렵습니다. 네이버 날씨나 기상청 날씨누리에서 **“부산 현재 습도”**를 검색해 보세요.
지금 부산은 **습도 약 79%**로 확인돼요. 기온은 약 **30°C**라서 꽤 후텁지근하게 느껴질 수 있습니다. ([yandex.com.tr](https://yandex.com.tr/hava/en/busan/month/august?utm_source=openai))


In [58]:
r.output

[ResponseReasoningItem(id='rs_056c83286fc2cec5006a850a0f0f30819ba5c8b42afe863586', summary=[], type='reasoning', content=[], encrypted_content='gAAAAABqhQoT5-lXQetv1oKT6XOd_8NzEIeVC_r2J3IhMqSldBHrk2co9nRNyp93adjIXsAGmDJ5JsLZgespmG_f_AlCp3af56qw6DADTctoC3iCVWbSmihzGeO4YmaALz4eq8INAquhCPwRTmNBuo1sCpfZYpvAJKABh0NsHbzb7NxYwWWEdavhazeB-vI2bRlgTb8IIM_fpUh5QUvz08aCUmoN53pShH3iRjM_Wo4tsM8ENLsu14X5tDy6g4UYsevOSwqcDnNZXVVZZl-9VrAJbPM_ooauyuqljgWV9NUPc0Eb030x4rzqUaUa8grWqtyPhN52ffKGW-sn1Klyw5vlz8Y2jU_jiyOlovrEaxfmM7Bi9CFtr64t2qXokz1FRRio1pN_yTaBYQg-PPoWC0W_qgbfswSnax2Dt0UwIGhlSXx5tQOdmUqP3v4gOCJw6SPy8gv3P3fMNJGr43VCY-v1KQ2Ir5fqWfnaZjNCfawFW4sofU-vJiOCRhA62TL-Xj2E6XXTKCzcUZWm9WTEjZNcvviRIUYNGbQAtnBfZ-3b8qeCPlyvCIBZQ1t_7WxI48owAucopWCmk498uZJ-QgpEBuITanF4m39vcM1PH8w9tLod98chFAxPuR8DdcfxG-JK_9TMnilHWs0wB89wtPbH2vlrowayJLf1TdVFOMeMi1r85CEuFvHAgdPr0MKwmckFavZHXLjAEb1P4TX1JTqZygpwKxOe0MmFD3Q4l9x-MbvN-bEBoK1uHEt06kW9xtzWg3HlIpgkkglB9OL0zSA8anbUcOnZo3su509nyIk_WOKdOeTHQY06zWyJR2e3UDz3Zjd75AEU8YWGpvtL5En2

In [ ]:
# 웹 검색 도구 과정이 포함되어있고, 웹 검색 결과를 통째로 입력값으로 받기 때문에
# inputs 토큰이 많이 늘어난다.
r.usage.input_tokens, r.usage.output_tokens, r.usage.total_tokens

(8665, 169, 8834)

In [64]:
# 코드 내장 도구
Q = "49029865 * 3212934 를 계산해서 숫자만 반환해줘."
ANSWER = 49029865 * 3212934

r = client.responses.create(model=API_MODEL,
                        input=Q,
                        tools=[{"type":"code_interpreter",
                                "container":{"type": "auto"}}]
                    )
print(r.output_text)
print(ANSWER)

157529720273910
157529720273910


In [ ]:
r.output  # 3단계의 추론 -> 도구 -> 출력

[ResponseReasoningItem(id='rs_001af828056ff20a006a8510c3ace887d08ed4ef362becace4', summary=[], type='reasoning', content=[], encrypted_content='gAAAAABqhRDKE63fJB4-OB7fnSo2kLAfPttJNpeE3BAPj12dN9sWCE2vGjaQXVEkKGBsF6q2rtVsmqPugpesLoD989SuXGa20Ud9u9B3FPgMylo9c-0hDAqkqvNnrdK6qiaVrrWCujSV2QvDRmn0qtRwODbe2E5xyVZHzB_clFG3rsOOd-gl_rBaIBzMhg-9XPLDUYLCIfJx1cy-xxjPQaKM4yxwHH2fikxuUeP1MJCc2iTWqffQ4L8c3iyvrUeaJVtBlZal6NtdlWei8Z8DkLKwcJOCCI6B2vH7uAST0woplA-85Dz9sWAalngylqwsioJ7sBgqsDDjTmqqZqz2BO4Q3Ru1ky_Ji6U7jH_PvoIBi4Y-shM7W8ycx6uWQCMudPnFmFW9Hp9hqMgALX0mq0JTfc-3f9u8adABPugOwhfp9wgEbCCl7md4Jup85DZJFvOMTK3cr5o_2AQn9OHKNS91hr-p35vdfDMX3TO7hEHkP9fxFhO9SstL0YulBkKZ6ohcd-rlMKRavI9udBEQ9_5ua1QwQXLSOlQY70wL6B4OoVYzflGuw9Z1vhxaIwt9EjToqrjTKV6OBg5SgRXQwyQVd4AGeYNpqqs1NbSo_6-YDq3X02RSzju6CW2HftrSFqWzC4jNEHQ3o17QIVjNyivwx274w5QEhPRbpOmsS47pZp13Sa1l98JSJoO8A_cBCzvb0QGESeojOvtWEAx6ghr2-UApxzU_yY4slV_dR3NaCK4dApStXRKqMmIEDWBvmkVPBod9RLRg_70X6ktS-CNMFnG55RFW40cuahV10ujvDvPn3g8WFNJu1sxzimSuh2YQ2t-dKPTxRn6fTpmRycwQ

In [67]:
r.usage.input_tokens, r.usage.output_tokens, r.usage.total_tokens

(1282, 41, 1323)